# agentic_alpha_long_short_crisis_alpha_search_v1

Diagnostic R-G-E-V-D card for Time Series Momentum, trend-following, and crisis-alpha ideas. It retrieves SOXL/SOXX/VIX through qlib `D.features`, evaluates long-short and crisis-overlay candidates against `qlib_paper_sim_v1`, records evidence with `R.start`, and keeps slim card sidecars.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.qlib_native import QlibProviderConfig, load_qlib_feature_frame
from src.factor import (
    build_long_short_crisis_alpha_frame,
    compact_long_short_crisis_alpha_summary,
    evaluate_long_short_crisis_alpha_candidates,
)
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_long_short_crisis_alpha_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
FEATURE_START = '2010-03-11'
FEATURE_END = '2026-06-10'
TEST_START = '2019-01-02'
TEST_END = '2026-06-09'
MAX_DRAWDOWN_FLOOR = -20.0
COST_PER_TURNOVER = 0.0002

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  feature_warmup:
    start: {FEATURE_START}
    end: {FEATURE_END}
  production_test:
    start: {TEST_START}
    end: {TEST_END}
rgevd:
  source: https://github.com/JacobDu/agentic-alpha
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  baseline_return_pct: {baseline.return_pct}
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use retrieve-generate-evaluate-validate-distill loops to close trend-following and crisis-alpha branches with comparable evidence.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'paper',
            'title': 'Time Series Momentum',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2089463',
            'extracted_signal': 'Test long recent winners and short recent losers using own past returns as the signal.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'A Century of Evidence on Trend-Following Investing',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2993026',
            'extracted_signal': 'Trend following can diversify traditional assets, but robustness must be tested in the target instrument and cost model.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Trend Following: Equity and Bond Crisis Alpha',
            'url': 'https://assets.super.so/e46b77e7-ee08-445e-b43f-4ffd88ae0a0e/files/96208c1d-28e0-412d-ad70-92499bbbd45a.pdf',
            'extracted_signal': 'Crisis-alpha framing motivates a negative or hedging sleeve when equity trend and volatility states deteriorate.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'twitter',
            'title': 'SOXL drawdown and crisis-alpha discussion seed',
            'url': 'https://x.com/ces921/highlights',
            'extracted_signal': 'SOXL path risk requires testing drawdown gates directly, not assuming trend-following crisis alpha transfers from diversified futures.',
            'retrieval_status': 'search_seed',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

provider_config = QlibProviderConfig(provider_uri=PROVIDER_URI, region='us', market='all', freq='day')
qlib_fields = ['$close', '$volume', '$vix']
qlib_frame = load_qlib_feature_frame(
    provider_config,
    instruments=['SOXL', 'SOXX'],
    fields=qlib_fields,
    start=FEATURE_START,
    end=FEATURE_END,
)
feature_frame = build_long_short_crisis_alpha_frame(qlib_frame, start=TEST_START, end=TEST_END)

candidate_metrics = evaluate_long_short_crisis_alpha_candidates(
    feature_frame,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    cost_per_turnover=COST_PER_TURNOVER,
)
candidate_metrics.to_csv(CARD / 'agentic_alpha_long_short_crisis_alpha_candidate_metrics.csv', index=False)

branches = sorted(candidate_metrics['branch'].unique())
compact_summary = compact_long_short_crisis_alpha_summary(
    candidate_metrics,
    branches=branches,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
passes_both = candidate_metrics.loc[candidate_metrics['beats_baseline_return'] & candidate_metrics['passes_drawdown_gate']]

search_summary = {
    'workflow': 'agentic_alpha_long_short_crisis_alpha_search',
    'stage_order': ['research', 'retrieve', 'generate', 'evaluate', 'validate', 'distill'],
    'baseline': {
        'name': baseline.name,
        'return_pct': float(baseline.return_pct),
        'max_drawdown_pct': float(baseline.max_drawdown_pct),
        'benchmark_return_pct': float(baseline.benchmark_return_pct),
        'recorder_id': baseline.recorder_id,
    },
    'goal_achieved': bool(compact_summary['goal_achieved']),
    'best_candidate': compact_summary['best_candidate'],
    'passes_both_count': int(compact_summary['passes_both_count']),
    'branch_results': compact_summary['branch_results'],
    'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
    'long_short_crisis_alpha': {
        'selected_features': [
            'SOXL_TREND_60',
            'SOXL_TREND_120',
            'SOXL_TREND_200',
            'SOXX_TREND_60',
            'SOXL_SOXX_RATIO_MOM_20',
            'SOXL_VIX',
            'SOXL_HV20',
        ],
        'candidate_coverage_start': feature_frame['datetime'].min().date().isoformat(),
        'candidate_coverage_end': feature_frame['datetime'].max().date().isoformat(),
        'source_coverage_start': feature_frame.attrs['source_coverage_start'],
        'source_coverage_end': feature_frame.attrs['source_coverage_end'],
        'feature_rows': int(len(feature_frame)),
        'qlib_provider': PROVIDER_URI.as_posix(),
        'qlib_fields': qlib_fields,
        'qlib_feature_rows': int(len(qlib_frame.reset_index())),
        'cost_per_turnover': COST_PER_TURNOVER,
    },
}
(CARD / 'agentic_alpha_long_short_crisis_alpha_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

failure_modes = (
    '# Failure Modes\n\n'
    '- long_short_time_series_momentum: SOXL/SOXX trend signals could not create a MaxDD<20% path; the best return setting stayed below the production baseline and still breached drawdown.\n'
    '- crisis_alpha_overlay: adding a negative crisis sleeve did not transfer diversified-futures crisis-alpha behavior to single-name leveraged semiconductor ETF exposure.\n'
    '- data_boundary: this card uses full production-window qlib SOXL/SOXX/VIX daily data through 2026-06-09 and therefore closes the branch on the comparable window.\n'
    '- next_action: move away from single-asset trend-following overlays; prioritize new orthogonal data such as longer option-risk history, capital-flow history, or model-family changes.\n'
)
(CARD / 'failure_modes.md').write_text(failure_modes, encoding='utf-8')

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        provider_uri=PROVIDER_URI.as_posix(),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        feature_start=FEATURE_START,
        feature_end=FEATURE_END,
        test_start=TEST_START,
        test_end=TEST_END,
        candidate_rows=len(candidate_metrics),
    )
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_long_short_crisis_alpha_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
    for path in [
        CARD / 'agentic_alpha_long_short_crisis_alpha_candidate_metrics.csv',
        CARD / 'agentic_alpha_long_short_crisis_alpha_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_long_short_crisis_alpha')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_artifacts'] = recorder_artifacts
(CARD / 'agentic_alpha_long_short_crisis_alpha_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'promotion_decision': 'not_promoted' if not search_summary['goal_achieved'] else 'promote_candidate',
    'as_of': TEST_END,
    'data_range': [TEST_START, TEST_END],
    'feature_warmup_range': [FEATURE_START, FEATURE_END],
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'recorder_artifacts': recorder_artifacts,
    'market': 'all',
    'region': 'us',
    'baseline': search_summary['baseline'],
    'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True},
    'branch_results': search_summary['branch_results'],
    'search_report': search_summary,
    'external_source_counts': search_summary['external_source_counts'],
    'long_short_crisis_alpha': search_summary['long_short_crisis_alpha'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'model': 'https://qlib.readthedocs.io/en/latest/component/model.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'workflow_config': 'workflow.yaml',
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_long_short_crisis_alpha_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'search_summary': 'agentic_alpha_long_short_crisis_alpha_search_summary.json',
    },
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'machine_entrypoint': 'summary.json',
        'primary_artifact': f'{EXPERIMENT}.ipynb',
        'kept_sidecars': [
            'agentic_alpha_long_short_crisis_alpha_candidate_metrics.csv',
            'agentic_alpha_long_short_crisis_alpha_search_summary.json',
            'external_sources.csv',
            'failure_modes.md',
        ],
        'recorder_only_artifacts': {
            'predictions': 'artifacts/pred.pkl',
            'labels': 'artifacts/label.pkl',
            'signals': 'artifacts/signals.pkl',
            'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
            'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
            'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
            'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        },
        'reproducible_from_recorder': True,
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'conclusion.md').write_text(
    '# 结论\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，把 Time Series Momentum、trend-following 和 crisis-alpha 外部知识映射到 SOXL/SOXX/VIX 的 long-short trend 与 crisis overlay 分支，并通过 qlib `D.features` 读取日线重新评估。\n'
    f'- key_metrics: production baseline return={baseline.return_pct:.2f}%, maxDD={baseline.max_drawdown_pct:.2f}%; 本卡 candidate_rows={len(candidate_metrics)}, passes_both_count={len(passes_both)}。\n'
    f'- regime_window_difference: 特征暖身使用 {FEATURE_START} 到 {FEATURE_END}，production-window 候选覆盖 {feature_frame["datetime"].min().date().isoformat()} 到 {feature_frame["datetime"].max().date().isoformat()}。\n'
    '- promotion_decision: not_promoted，未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。\n'
    '- risks: 单资产 SOXL 的 trend-following/short crisis sleeve 没有复制 diversified futures 的 crisis alpha；做空或对冲腿增加换手和反转损失，仍无法压住路径回撤。\n'
    '- next_action: 下一轮不要继续扩展单资产 trend-following 阈值，优先尝试真正正交的数据或模型族。\n',
    encoding='utf-8',
)
print(json.dumps({'recorder_id': recorder_id, 'passes_both_count': int(len(passes_both)), 'branch_results': search_summary['branch_results']}, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[55319:MainThread](2026-06-23 18:08:38,301) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55319:MainThread](2026-06-23 18:08:38,303) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55319:MainThread](2026-06-23 18:08:38,303) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[55319:MainThread](2026-06-23 18:08:38,416) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_long_short_crisis_alpha_search_v1.


[55319:MainThread](2026-06-23 18:08:38,418) INFO - qlib.workflow - [exp.py:258] - Experiment 650154366397263200 starts running ...


[55319:MainThread](2026-06-23 18:08:38,486) INFO - qlib.workflow - [recorder.py:345] - Recorder 1d4adcc6907b4271ad404e3b6232ff97 starts running under Experiment 650154366397263200 ...


[55319:MainThread](2026-06-23 18:08:38,576) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{
  "branch_results": {
    "crisis_alpha_overlay": {
      "best_drawdown_gate_available": false,
      "best_drawdown_gate_candidate": "soxl_soxx_dual_base-0.05_b2.0_h0.5_tv0.4",
      "best_drawdown_gate_max_drawdown_pct": -73.49488005876239,
      "best_drawdown_gate_return_pct": 2469.4272209816054,
      "best_return_candidate": "soxl_soxx_dual_base-0.05_b2.0_h0.5_tv0.4",
      "best_return_max_drawdown_pct": -73.49488005876239,
      "best_return_pct": 2469.4272209816054,
      "coverage_end": "2026-06-09",
      "coverage_start": "2019-01-02",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 144,
      "max_drawdown_floor": -20.0,
      "passes_both_count": 0
    },
    "long_short_time_series_momentum": {
      "best_drawdown_gate_available": false,
      "best_drawdown_gate_candidate": "soxl_soxx_dual_p-0.05_n0.0_none_l1.0_s0.0_tvNone",
      "best_drawdown_gate_max_drawdown_pct": -73.57013328956272,
      "best_drawdown_gate_return_pct": 3963.658227776891

## Final Chinese Conclusion

The executed cell writes `conclusion.md` with the required Chinese summary.